# 01 — Análise Exploratória de Dados (EDA)

Este notebook explora o dataset sintético de voos domésticos brasileiros.  
O objetivo é compreender a distribuição das variáveis, identificar padrões,
tratar valores ausentes e gerar insights relevantes para a modelagem.

**Dataset:** 5.000 registros de voos gerados com `src/utils.py`  
**Variável alvo:** `delayed` — indica se o voo chegou com mais de 15 minutos de atraso

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

from src.utils import generate_flight_data

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

df = generate_flight_data(n_samples=5000, random_state=42)
print(f"Shape: {df.shape}")
df.head()

## 1. Visão Geral do Dataset

In [ ]:
print("=== Tipos de dados ===")
print(df.dtypes)
print("\n=== Valores ausentes por coluna ===")
print(df.isnull().sum())

In [ ]:
# Estatísticas descritivas para variáveis numéricas
df.describe(include='all').T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

## 2. Análise de Valores Ausentes

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'ausentes': missing, 'porcentagem': missing_pct})
missing_df = missing_df[missing_df['ausentes'] > 0]
print("Colunas com valores ausentes:")
print(missing_df)
print("\nOs valores ausentes em dep_delay_min, arr_delay_min e delayed")
print(f"correspondem aos {df['cancelled'].sum()} voos cancelados ({df['cancelled'].mean()*100:.1f}% do total).")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
missing_df['porcentagem'].plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('% de valores ausentes')
ax.set_title('Percentual de Valores Ausentes por Coluna')
for i, v in enumerate(missing_df['porcentagem']):
    ax.text(v + 0.05, i, f'{v}%', va='center')
plt.tight_layout()
plt.show()

**Insight:** Os valores ausentes nas colunas de atraso são estruturais — ocorrem apenas para voos cancelados, representando ~3% do dataset. Estratégia de tratamento: remover os registros cancelados antes da modelagem.

## 3. Distribuição da Variável Alvo

In [ ]:
df_clean = df[df['cancelled'] == 0].copy()
target_counts = df_clean['delayed'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

target_counts.plot(kind='bar', ax=axes[0], color=['#2196F3', '#F44336'], edgecolor='black')
axes[0].set_xticklabels(['Não Atrasado (0)', 'Atrasado (1)'], rotation=0)
axes[0].set_title('Distribuição da Variável Alvo')
axes[0].set_ylabel('Contagem')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2, p.get_height()+10), ha='center')

axes[1].pie(target_counts, labels=['Não Atrasado', 'Atrasado'],
            autopct='%1.1f%%', colors=['#2196F3', '#F44336'], startangle=90)
axes[1].set_title('Proporção da Variável Alvo')
plt.tight_layout()
plt.show()

print(f"Classe 0 (não atrasado): {target_counts[0]} ({target_counts[0]/len(df_clean)*100:.1f}%)")
print(f"Classe 1 (atrasado):     {target_counts[1]} ({target_counts[1]/len(df_clean)*100:.1f}%)")

## 4. Distribuição de Variáveis Numéricas

In [ ]:
num_cols = ['dep_delay_min', 'arr_delay_min', 'distance_km', 'taxi_out_min', 'taxi_in_min', 'carrier_delay_min']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df_clean[col], ax=axes[i], kde=True, color='steelblue', bins=40)
    axes[i].set_title(col)
    axes[i].set_xlabel('')

plt.suptitle('Distribuição das Variáveis Numéricas', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Análise de Variáveis Categóricas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Atraso por companhia aérea
delay_by_airline = df_clean.groupby('airline')['delayed'].mean().sort_values(ascending=False) * 100
delay_by_airline.plot(kind='bar', ax=axes[0], color='coral', edgecolor='black')
axes[0].set_title('Taxa de Atraso por Companhia Aérea (%)')
axes[0].set_ylabel('Taxa de Atraso (%)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width()/2, p.get_height()+0.3), ha='center', fontsize=9)

# Atraso por condição climática
delay_by_weather = df_clean.groupby('weather')['delayed'].mean().sort_values(ascending=False) * 100
delay_by_weather.plot(kind='bar', ax=axes[1], color='skyblue', edgecolor='black')
axes[1].set_title('Taxa de Atraso por Condição Climática (%)')
axes[1].set_ylabel('Taxa de Atraso (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width()/2, p.get_height()+0.3), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Atraso por dia da semana
day_labels = ['Seg', 'Ter', 'Qua', 'Qui', 'Sex', 'Sáb', 'Dom']
delay_by_dow = df_clean.groupby('day_of_week')['delayed'].mean() * 100
delay_by_dow.index = day_labels
delay_by_dow.plot(kind='bar', ax=axes[0], color='mediumseagreen', edgecolor='black')
axes[0].set_title('Taxa de Atraso por Dia da Semana (%)')
axes[0].set_ylabel('Taxa de Atraso (%)')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Atraso por mês
delay_by_month = df_clean.groupby('month')['delayed'].mean() * 100
month_labels = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
delay_by_month.index = month_labels
delay_by_month.plot(kind='bar', ax=axes[1], color='mediumpurple', edgecolor='black')
axes[1].set_title('Taxa de Atraso por Mês (%)')
axes[1].set_ylabel('Taxa de Atraso (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

## 6. Análise de Correlação

In [ ]:
corr_cols = ['dep_delay_min', 'arr_delay_min', 'distance_km', 'taxi_out_min',
             'taxi_in_min', 'carrier_delay_min', 'day_of_week', 'month', 'dep_hour', 'delayed']
corr_matrix = df_clean[corr_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Mapa de Correlação entre Variáveis Numéricas')
plt.tight_layout()
plt.show()

In [ ]:
# Correlação com a variável alvo
target_corr = df_clean[corr_cols].corr()['delayed'].drop('delayed').sort_values(ascending=False)
plt.figure(figsize=(8, 4))
target_corr.plot(kind='bar', color=['#F44336' if v > 0 else '#2196F3' for v in target_corr])
plt.title('Correlação das Features com a Variável Alvo (delayed)')
plt.ylabel('Coeficiente de Pearson')
plt.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()
print(target_corr)

## 7. Análise de Outliers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, ['dep_delay_min', 'arr_delay_min', 'distance_km']):
    sns.boxplot(data=df_clean, x='delayed', y=col, ax=ax, palette=['#2196F3','#F44336'])
    ax.set_xticklabels(['Não Atrasado', 'Atrasado'])
    ax.set_title(f'{col} por classe de atraso')

plt.suptitle('Boxplots: Variáveis numéricas vs. Variável Alvo', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Análise Temporal — Atraso por Horário de Partida

In [ ]:
delay_by_hour = df_clean.groupby('dep_hour')['delayed'].mean() * 100

plt.figure(figsize=(12, 4))
delay_by_hour.plot(marker='o', color='steelblue', linewidth=2)
plt.fill_between(delay_by_hour.index, delay_by_hour.values, alpha=0.2, color='steelblue')
plt.title('Taxa de Atraso (%) por Horário de Partida')
plt.xlabel('Hora de Partida')
plt.ylabel('Taxa de Atraso (%)')
plt.xticks(range(5, 24))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Insights da EDA

Com base na análise exploratória, podemos destacar os seguintes **insights**:

1. **Valores ausentes estruturais**: as colunas `dep_delay_min`, `arr_delay_min` e `delayed` apresentam ~3% de valores ausentes, todos correspondentes a voos cancelados. A estratégia adequada é remover esses registros antes da modelagem.

2. **Desbalanceamento de classes**: aproximadamente 35–40% dos voos são classificados como atrasados. O dataset não apresenta desbalanceamento crítico, mas métricas como F1-Score e AUC-ROC são preferíveis à acurácia simples.

3. **Condições climáticas são determinantes**: voos em condições de `STORM` apresentam a maior taxa de atraso, seguidos por `RAIN` e `FOG`, confirmando que o clima é a variável com maior impacto no atraso.

4. **Atraso na partida é altamente correlacionado com atraso na chegada**: a correlação entre `dep_delay_min` e `arr_delay_min` é muito alta (≥ 0.90), indicando que o atraso se propaga ao longo do voo.

5. **Padrão temporal**: voos noturnos e de final de semana tendem a ter maior taxa de atraso, possivelmente pela acumulação de atrasos ao longo do dia.

6. **Distância não é determinante isoladamente**: a correlação entre `distance_km` e `delayed` é fraca, indicando que a distância não é um preditor forte de atraso por si só.